# Escalabilidade Algoritmos A* Paralelos
### Análise do ganho de desempenho por aumento de threads
---

> **Sobre este notebook**
> Analisa como cada abordagem paralela do A* se comporta conforme o número de threads aumenta.
> A comparação é feita **dentro de cada abordagem** (ex: HDA* 4 vs 6 vs 8 vs 10 threads)
> e **entre abordagens** no mesmo número de threads.
>
> **Grid analisado:** 5.000×5.000 células
> **Referência base:** A* Serial (1 thread)

Instalando LIBS e configuração

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'legend.framealpha': 0.9,
})

# variaveis globais
OBS_VALS = [0, 10, 20, 30]
OBS_LBLS = ['0%', '10%', '20%', '30%']
GRID_5K  = '5000x5000'

AB_COLORS = {
    'Serial':          '#2c3e50',
    'HDA*':            '#e67e22',
    'Bidirecional A*': '#27ae60',
    'PBNF':            '#e74c3c',
}
THREAD_COLORS = {
    1:  '#95a5a6',
    4:  '#3498db',
    6:  '#9b59b6',
    8:  '#e67e22',
    10: '#e74c3c',
    12: '#1abc9c',
}


### Caminhos dos arquivos CSV

Coloque todos os arquivos na mesma pasta deste notebook.

In [ ]:
CSV_SERIAL = 'serial.csv'

CSV_HDA_4  = 'parallel_hda.csv'
CSV_HDA_6  = 'parallel_hda_6threads.csv'
CSV_HDA_8  = 'parallel_hda_8threads.csv'
CSV_HDA_10 = 'parallel_hda_10threads.csv'
CSV_HDA_12 = None 

CSV_BIDIR_4  = 'parallel_bidirectional.csv'
CSV_BIDIR_6  = None
CSV_BIDIR_8  = None
CSV_BIDIR_10 = None
CSV_BIDIR_12 = None

CSV_PBNF_4  = 'parallel_pbnf.csv'
CSV_PBNF_6  = None
CSV_PBNF_8  = None
CSV_PBNF_10 = None
CSV_PBNF_12 = None


In [ ]:
# Mapeia cada variável de caminho para (abordagem, n_threads, separador)

_FILE_MAP = [
    (CSV_SERIAL,   'Serial',          1),
    (CSV_HDA_4,    'HDA*',            4),
    (CSV_HDA_6,    'HDA*',            6),
    (CSV_HDA_8,    'HDA*',            8),
    (CSV_HDA_10,   'HDA*',           10),
    (CSV_HDA_12,   'HDA*',           12),
    (CSV_BIDIR_4,  'Bidirecional A*', 4),
    (CSV_BIDIR_6,  'Bidirecional A*', 6),
    (CSV_BIDIR_8,  'Bidirecional A*', 8),
    (CSV_BIDIR_10, 'Bidirecional A*',10),
    (CSV_BIDIR_12, 'Bidirecional A*',12),
    (CSV_PBNF_4,   'PBNF',            4),
    (CSV_PBNF_6,   'PBNF',            6),
    (CSV_PBNF_8,   'PBNF',            8),
    (CSV_PBNF_10,  'PBNF',           10),
    (CSV_PBNF_12,  'PBNF',           12),
]

def _detect_sep(path):
    with open(path, 'r', encoding='utf-8') as f:
        first_line = f.readline()
    return ';' if first_line.count(';') >= first_line.count(',') else ','

def _load(path, abordagem, n_threads):
    sep = _detect_sep(path)
    df  = pd.read_csv(path, sep=sep)
    df.columns        = df.columns.str.strip()
    df['abordagem']   = abordagem
    df['n_threads']   = n_threads
    df['grid_size']   = df['Tamanho do Grid'].str.extract(r'(\d+x\d+)')[0]
    df['obst_pct']    = df['Percentual Obstaculos (%)'].astype(int)
    df['tempo_ms']    = pd.to_numeric(df['Tempo de Execucao (ms)'],          errors='coerce')
    df['nos_exp']     = pd.to_numeric(df['Nos Expandidos'],                   errors='coerce')
    df['custo_total'] = pd.to_numeric(df['Custo Total do Caminho'],           errors='coerce')
    return df

frames = []
skipped = []
for path, abordagem, n_threads in _FILE_MAP:
    if path is None:
        skipped.append(f'{abordagem} {n_threads}t')
        continue
    try:
        frames.append(_load(path, abordagem, n_threads))
    except FileNotFoundError:
        skipped.append(f'{abordagem} {n_threads}t (arquivo não encontrado: {path})')

raw   = pd.concat(frames, ignore_index=True)
raw5k = raw[raw['grid_size'] == GRID_5K].copy()

# Agregação 
agg = raw5k.groupby(['abordagem', 'n_threads', 'obst_pct']).agg(
    tempo_medio = ('tempo_ms',    'mean'),
    tempo_std   = ('tempo_ms',    'std'),
    tempo_cv    = ('tempo_ms',    lambda x: x.std() / x.mean() * 100),
    nos_medio   = ('nos_exp',     'mean'),
    custo_medio = ('custo_total', 'mean'),
).reset_index()

serial_ref = (agg[agg['abordagem'] == 'Serial']
              [['obst_pct', 'tempo_medio']]
              .rename(columns={'tempo_medio': 'tempo_serial'}))
agg = agg.merge(serial_ref, on='obst_pct', how='left')
agg['speedup']    = agg['tempo_serial'] / agg['tempo_medio']
agg['eficiencia'] = agg['speedup']      / agg['n_threads']

base_ref = (agg[agg.groupby('abordagem')['n_threads']
                .transform('min') == agg['n_threads']]
            [['abordagem', 'obst_pct', 'tempo_medio']]
            .rename(columns={'tempo_medio': 'tempo_base_threads'}))
agg = agg.merge(base_ref, on=['abordagem', 'obst_pct'], how='left')
agg['speedup_intra'] = agg['tempo_base_threads'] / agg['tempo_medio']

paralelas_disponiveis = sorted([
    ab for ab in agg['abordagem'].unique() if ab != 'Serial'
])

# resumo dos dados carregados 
print("✅ Dados carregados — Grid 5K×5K")
print(f"   {len(raw5k)} execuções individuais\n")
print("   Threads disponíveis por abordagem:")
inv = agg[agg['abordagem'] != 'Serial'].groupby('abordagem')['n_threads'].apply(sorted)
for ab, threads in inv.items():
    print(f"     {ab}: {list(threads)} threads")
if skipped:
    print(f"\n   ⏭️  Ignorados (None ou não encontrados):")
    for s in skipped:
        print(f"     - {s}")


## Seção - Escalabilidade por Abordagem

Como o **tempo de execução** varia conforme o número de threads aumenta,
separado por densidade de obstáculos.

As linhas horizontais tracejadas indicam o tempo do **Serial** em cada cenário.


In [ ]:
w_ab1  = widgets.Dropdown(
    options=paralelas_disponiveis,
    description='Abordagem:',
    style={'description_width': '90px'}
)
w_log1 = widgets.Checkbox(value=False, description='Escala logarítmica')
out1   = widgets.Output()

def plot_escala_intra(*_):
    with out1:
        clear_output(wait=True)
        ab      = w_ab1.value
        use_log = w_log1.value
        sub     = agg[agg['abordagem'] == ab].sort_values('n_threads')
        threads = sorted(sub['n_threads'].unique())
        serial  = agg[agg['abordagem'] == 'Serial'].sort_values('obst_pct')

        fig, ax = plt.subplots(figsize=(9, 5))

        for obst in OBS_VALS:
            cor = plt.cm.RdYlGn_r(obst / 35)
            d   = sub[sub['obst_pct'] == obst]
            if d.empty: continue
            ax.errorbar(
                d['n_threads'], d['tempo_medio'], yerr=d['tempo_std'],
                label=f'{obst}% obstáculos',
                color=cor, marker='o', linewidth=2.2,
                markersize=8, capsize=4
            )
            s_val = serial[serial['obst_pct'] == obst]['tempo_medio'].values
            if len(s_val):
                ax.axhline(s_val[0], linestyle='--', linewidth=1,
                           color=cor, alpha=0.4)
                ax.annotate(f'Serial {obst}%: {s_val[0]:,.0f}ms',
                            xy=(threads[-1], s_val[0]),
                            xytext=(5, 0), textcoords='offset points',
                            fontsize=7.5, color=cor, va='center')

        if use_log:
            ax.set_yscale('log')
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

        ax.set_xticks(threads)
        ax.set_xticklabels([f'{t}t' for t in threads])
        ax.set_xlabel('Número de Threads')
        ax.set_ylabel('Tempo Médio de Execução (ms)')
        ax.set_title(f'Escalabilidade de Threads — {ab} · Grid 5K×5K')
        ax.legend(title='Obstáculos', fontsize=9)
        plt.tight_layout()
        plt.show()

for w in [w_ab1, w_log1]:
    w.observe(plot_escala_intra, names='value')

display(widgets.HBox([w_ab1, w_log1]), out1)
plot_escala_intra()

## Seção 2 Speedup entre Abordagem

Ganho de velocidade **dentro da própria abordagem** conforme threads aumentam,
normalizado em relação ao menor número de threads disponível (base = 1×).

> Um speedup linear seria o ideal teórico: dobrar threads = dobrar velocidade.


In [ ]:
w_ab2 = widgets.Dropdown(
    options=paralelas_disponiveis,
    description='Abordagem:',
    style={'description_width': '90px'}
)
out2 = widgets.Output()

def plot_speedup_intra(*_):
    with out2:
        clear_output(wait=True)
        ab      = w_ab2.value
        sub     = agg[agg['abordagem'] == ab].sort_values('n_threads')
        threads = sorted(sub['n_threads'].unique())
        base_t  = threads[0]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(
            f'Speedup Intra-Abordagem — {ab} (base = {base_t} threads) · Grid 5K×5K',
            fontsize=13, fontweight='bold'
        )

        # speedup por obstáculo
        ax = axes[0]
        for obst in OBS_VALS:
            d = sub[sub['obst_pct'] == obst]
            if d.empty: continue
            ax.plot(d['n_threads'], d['speedup_intra'],
                    label=f'{obst}% obs.',
                    color=plt.cm.RdYlGn_r(obst / 35),
                    marker='o', linewidth=2.2, markersize=8)
            last = d.sort_values('n_threads').iloc[-1]
            ax.annotate(f"{last['speedup_intra']:.2f}×",
                        xy=(last['n_threads'], last['speedup_intra']),
                        xytext=(5, 2), textcoords='offset points', fontsize=8)

        ideal_x = np.array(threads, dtype=float)
        ideal_y = ideal_x / base_t
        ax.plot(ideal_x, ideal_y, linestyle='--', color='gray',
                linewidth=1.4, label='Ideal (linear)', alpha=0.6)
        ax.axhline(1, linestyle=':', color='gray', linewidth=1)
        ax.set_xticks(threads)
        ax.set_xticklabels([f'{t}t' for t in threads])
        ax.set_xlabel('Número de Threads')
        ax.set_ylabel(f'Speedup (relativo a {base_t} threads)')
        ax.set_title('Speedup por densidade de obstáculos')
        ax.legend(fontsize=9)

        # eficiência de escalonamento
        ax = axes[1]
        for obst in OBS_VALS:
            d = sub[sub['obst_pct'] == obst]
            if d.empty: continue
            efic = d['speedup_intra'] / (d['n_threads'] / base_t)
            ax.plot(d['n_threads'], efic,
                    label=f'{obst}% obs.',
                    color=plt.cm.RdYlGn_r(obst / 35),
                    marker='o', linewidth=2.2, markersize=8)

        ax.axhline(1, linestyle='--', color='gray', linewidth=1.4,
                   label='Eficiência ideal (100%)')
        ax.set_xticks(threads)
        ax.set_xticklabels([f'{t}t' for t in threads])
        ax.set_xlabel('Número de Threads')
        ax.set_ylabel('Eficiência de Escalonamento')
        ax.set_title('Eficiência (Speedup / Fator de threads)')
        ax.legend(fontsize=9)

        plt.tight_layout()
        plt.show()

w_ab2.observe(plot_speedup_intra, names='value')
display(w_ab2, out2)
plot_speedup_intra()